In [1]:
import random
import math
import csv
import pandas as pd

def generate_partition_array(n, m=1000):
    """
    n: 전체 원소 개수 (s1 + s2)
    m: 개별 원소의 최대 허용값 (기본값 1000)
    A: s1을 uniform 샘플링한 뒤 그 합으로 결정됨
    """

    def break_stick(target_sum, count, max_val):
        # s2 생성: target_sum을 count개의 조각으로 나누는 로직 (Stick Breaking)
        # 원소 범위: [1, max_val]
        attempts = 0
        while attempts < 10000:
            points = sorted(random.sample(range(1, target_sum), count - 1))
            points = [0] + points + [target_sum]
            pieces = [points[i+1] - points[i] for i in range(count)]
            if all(p <= max_val for p in pieces):
                return pieces
            attempts += 1
        return None  # 실패 시 None 반환

    # s1을 '더 작은 집합'으로 둠
    #   - k < n-k          ->  k <= (n-1)//2     (s1이 더 작은 집합)
    #   - s1도 n/5 보다 많음 ->  k >= n//5 + 1
    min_k = n // 5 + 1
    max_k = (n - 1) // 2

    while True:
        # 1. k 결정
        k = random.randint(min_k, max_k)

        # 2. s1 생성: [250, 500] 정수 균등 분포에서 k개 독립 샘플링
        s1 = [random.randint(250, 500) for _ in range(k)]
        A = sum(s1)

        # 불가능 구간이면 break_stick 호출 없이 즉시 재추첨
        if not ((n - k) * 1 <= A <= (n - k) * 1000):
            continue

        # 3. s2 생성: 원소 범위 [1, 1000], 실패 시 k부터 다시 뽑음
        s2 = break_stick(A, n - k, max_val=1000)
        if s2 is not None:
            break

    print(f"k={k}, n-k={n-k}, A={A}")
    print(f"s1 ({k}개): {s1}")
    print(f"s2 ({n-k}개): {s2}")
    print(f"s1합: {sum(s1)}")
    print(f"s2합: {sum(s2)}")

    # 4. 두 집합 병합
    instance = s1 + s2

    return instance


In [2]:
def apply_johnsons_rule_and_save(g, c1, w1, c2, w2, nnn):
    k = 1  # 수식 내 파라미터, 차피 순서 안바뀌니 1로 고정해서 계산
    jobs = []

    # 1. 수식 적용: p_ij = ((c_ij * w_ij) / k)^(k/(k+1))
    for j in range(1, g + 2):
        p1_val = (c1[j] * w1[j] / k) ** (k / (k + 1))
        p2_val = (c2[j] * w2[j] / k) ** (k / (k + 1))

        jobs.append({
            'JOB_ID': f"j{j-1}",
            'p1': p1_val,
            'p2': p2_val,
            'C1': c1[j], 'W1': w1[j],
            'C2': c2[j], 'W2': w2[j]
        })

    # 2. 기계 1과 2의 시간이 같으면 앞에 배치 (p1 <= p2)
    group1 = [j for j in jobs if j['p1'] <= j['p2']]
    group2 = [j for j in jobs if j['p1'] > j['p2']]

    # Group 1: p1 기준 오름차순
    group1.sort(key=lambda x: x['p1'])
    # Group 2: p2 기준 내림차순
    group2.sort(key=lambda x: x['p2'], reverse=True)

    sorted_jobs = group1 + group2

    # 3. CSV 저장
    filename_sorted = f"2026ADP_Partition_Tgp1_2t2z_{g+1}_N_{nnn:02d}_inst_input.csv"
    df = pd.DataFrame(sorted_jobs)
    final_df = df[['JOB_ID', 'C1', 'W1', 'C2', 'W2']]
    final_df.columns = ['JOB_ID', 'JOB_C1', 'JOB_W1', 'JOB_C2', 'JOB_W2']

    final_df.to_csv(filename_sorted, index=False)
    print(f"재배열 완료 (p1=p2 우선배치 적용): {filename_sorted}")


In [3]:
def transform_to_micp_instance(partition_instance, nnn):
    """
    partition_instance: 앞서 생성한 a_j들의 배열 (g개의 원소)
    """
    g = len(partition_instance)
    a = partition_instance

    # 1. A값 계산 (Partition 문제의 목표 합)
    A = sum(a) / 2

    # 2. b_j 값 생성 (j = 1, ..., g+1)
    b = [0] * (g + 2)
    for j in range(1, g + 1):
        b[j] = a[j-1]

    b[g+1] = 2 * A

    # 3. 파라미터 delta와 M 계산
    def calculate_delta(A):
        def f(x):
            term1 = math.sqrt(13 * (A**2) - 10 * A * x + 2 * (x**2))
            term2 = math.sqrt(13 * (A**2) + 10 * A * x + 2 * (x**2))
            return 2*term1 + 2*term2
        f_0 = f(0)
        f_1 = f(1)
        delta = min(1.0, f_1 - f_0)
        return delta

    delta = calculate_delta(A)
    M = 8 * math.sqrt(A) / delta

    # 4. w_i,j 및 c_i,j 생성
    w1 = [0.0] * (g + 2)
    w2 = [0.0] * (g + 2)
    c1 = [0.0] * (g + 2)
    c2 = [0.0] * (g + 2)

    for j in range(1, g + 1):
        w1[j] = b[j]
        w2[j] = b[j]    # 머신 2: b[j]
        c1[j] = b[j]
        c2[j] = b[j]    # 머신 2: b[j]

    help_t = 0.2
    help_z = 2.0

    w1[g+1] = help_z * A
    w2[g+1] = help_z * A
    c1[g+1] = help_z * A
    c2[g+1] = help_z * A

    filename = f"2026ADP_Partition_savedTgp1_2t2z_{g+1}_N_{nnn:02d}_inst.csv"
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['JOB_ID', 'JOB_C1', 'JOB_W1', 'JOB_C2', 'JOB_W2'])
        for j in range(1, g + 2):
            job_id = f"j{j-1}"
            writer.writerow([job_id, c1[j], w1[j], c2[j], w2[j]])

    print(f"CSV 파일 저장 완료: {filename}")

    apply_johnsons_rule_and_save(g, c1, w1, c2, w2, nnn)


In [ ]:
NNN = [300,400,500]
inputm = 1000

for N in NNN:
    inputn = N - 1
    for nnn in range(1, 31):
        dd = generate_partition_array(n=inputn, m=inputm)
        transform_to_micp_instance(partition_instance=dd, nnn=nnn)


k=5, n-k=4, A=1780
s1 (5개): [318, 391, 299, 317, 455]
s2 (4개): [197, 689, 682, 212]
s1합: 1780
s2합: 1780
CSV 파일 저장 완료: 2026ADP_Partition_savedTgp1_2t2z_10_N_01_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Partition_Tgp1_2t2z_10_N_01_inst_input.csv
k=5, n-k=4, A=1946
s1 (5개): [451, 389, 250, 370, 486]
s2 (4개): [856, 180, 622, 288]
s1합: 1946
s2합: 1946
CSV 파일 저장 완료: 2026ADP_Partition_savedTgp1_2t2z_10_N_02_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Partition_Tgp1_2t2z_10_N_02_inst_input.csv
k=5, n-k=4, A=1770
s1 (5개): [315, 381, 420, 373, 281]
s2 (4개): [841, 467, 82, 380]
s1합: 1770
s2합: 1770
CSV 파일 저장 완료: 2026ADP_Partition_savedTgp1_2t2z_10_N_03_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Partition_Tgp1_2t2z_10_N_03_inst_input.csv
k=6, n-k=3, A=2433
s1 (6개): [311, 439, 313, 459, 456, 455]
s2 (3개): [762, 1000, 671]
s1합: 2433
s2합: 2433
CSV 파일 저장 완료: 2026ADP_Partition_savedTgp1_2t2z_10_N_04_inst.csv
재배열 완료 (p1=p2 우선배치 적용): 2026ADP_Partition_Tgp1_2t2z_10_N_04_inst_input.csv
k=5, n-k=4, A=1910
s1 (5개): 